In [ ]:
!pip install -q google-genai tqdm pandas rapidfuzz

# config

In [ ]:
import re
import os
import json
import time
import uuid
import logging
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz

from google import genai
from google.genai import types
from google.colab import userdata, files


logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


# input file
BOOK_PATH = Path("/content/full_DVSK_data.json")

CHUNK_FILES = {
    "chunk_256": Path("/content/full_DVSK_data_bkai_256.json"),
    "chunk_512": Path("/content/full_DVSK_data_aiteamvn_512.json"),
    "chunk_1024": Path("/content/full_DVSK_data_aiteamvn_1024.json"),
}


# output file
OUTPUT_JSONL = Path("/content/dvsk_embedding_benchmark_qrels.jsonl")
OUTPUT_JSON = Path("/content/dvsk_embedding_benchmark_qrels.json")
OUTPUT_CSV = Path("/content/dvsk_embedding_benchmark_qrels.csv")
FAILED_JSON = Path("/content/failed_claim_windows.json")



GEMINI_MODEL = "gemini-3.1-flash-lite"
TARGET_CLAIMS = 300

PAGES_PER_WINDOW = 2
MAX_CLAIMS_PER_WINDOW = 4
SLEEP_SECONDS = 3


# match config
MIN_EXACT_LEN = 30
FUZZY_THRESHOLD = 78
TOP_MATCHES_PER_CHUNK_SIZE = 3

## sách

In [ ]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


book_data = load_json(BOOK_PATH)

book_name = book_data["book_name"]
full_text = book_data["full_text"]
page_footnotes_map = book_data.get("page_footnotes_map", {})

chunk_data = {
    key: load_json(path)
    for key, path in CHUNK_FILES.items()
}


print("Book:", book_name)
print("Chunk files:")
for key, rows in chunk_data.items():
    print(key, len(rows))
# print(len(chunk_data))

# tách nội dung sách ra thành page text
def split_pages(full_text: str) -> List[Dict[str, Any]]:
    pattern = r"\n\s*\[\[\[PAGE:(\d+)\]\]\]\s*\n"
    matches = list(re.finditer(pattern, full_text))

    pages = []

    for i, match in enumerate(matches):
        page_num = int(match.group(1))
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)

        text = full_text[start:end].strip()

        if text:
            pages.append({
                "page": page_num,
                "text": text,
            })

    return pages


pages = split_pages(full_text)

print("Total pages:", len(pages))
print(pages[0]["page"])
print(pages[0]["text"][:500])


def normalize_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    text = text.replace(" ,", ",").replace(" .", ".")
    return text.strip()


def remove_page_marker(text: str) -> str:
    return re.sub(r"\n\s*\[\[\[PAGE:(\d+)\]\]\]\s*\n", " ", text)


def clean_for_match(text: str) -> str:
    text = remove_page_marker(text)
    text = text.lower()
    text = re.sub(r"\[\d+\]", " ", text)
    text = re.sub(r"[^0-9a-zA-ZÀ-ỹ\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def short_text(text: str, max_len: int = 500) -> str:
    text = normalize_text(text)
    return text[:max_len] + "..." if len(text) > max_len else text



def make_page_windows(
    pages: List[Dict[str, Any]],
    pages_per_window: int = 2,
) -> List[Dict[str, Any]]:

    windows = []

    for i in range(0, len(pages), pages_per_window):
        group = pages[i:i + pages_per_window]

        if not group:
            continue

        text = "\n\n".join(
            f"[[[PAGE:{p['page']}]]]\n{p['text']}"
            for p in group
        )

        page_nums = [p["page"] for p in group]

        windows.append({
            "window_id": f"window_{len(windows) + 1:05d}",
            "book_name": book_name,
            "pages": page_nums,
            "text": text,
            "footnotes": {
                str(p): page_footnotes_map.get(str(p), {})
                for p in page_nums
            }
        })

    return windows


windows = make_page_windows(pages, PAGES_PER_WINDOW)

print("Total windows:", len(windows))
print(windows[0]["window_id"], windows[0]["pages"])
print(windows[0]["text"][:1000])

Book: Đại Việt Sử Ký Toàn Thư
Chunk files:
chunk_256 1003
chunk_512 682
chunk_1024 460
Total pages: 154
154
việc lớn của nước, quốc sử không thể không chép được. Có lẽ Lê Văn Hưu thấy đều gọi là Thọ lăng, cho là không đúng lễ nên bỏ đi, nhưng thế không phải là phép làm sử.
Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự. Lại đem thuyền rồng đi đón Trần thị. Anh Trần thị là Trần Tự Khánh cho rằng bấy giờ đương lúc loạn lạc, chưa đưa đi ngay được.
Sai sứ cáo phó với nhà Tống, nhà Tống sai người sang làm lễ tế điếu.
Huệ Tông Hoàng Đ
Total windows: 77
window_00001 [154, 155]
[[[PAGE:154]]]
việc lớn của nước, quốc sử không thể không chép được. Có lẽ Lê Văn Hưu thấy đều gọi là Thọ lăng, cho là không đúng lễ nên bỏ đi, nhưng thế không phải là phép làm sử.
Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự. Lại đem thuyền rồng đi đón Trần thị. Anh Trần 

## prompt

``` Input: Một "window" (nội dung của 2 trang sách).

Prompt: Đóng gói nội dung vào prompt cùng với yêu cầu tạo "claim".

Interaction: Gửi prompt tới Gemini, nếu lỗi thì tự động retry.

Parse: Lấy JSON từ phản hồi của Gemini.

Clean: Xử lý dữ liệu thô, gán ID, gắn nhãn metadata.

Output: Một danh sách các "cặp" (Claim - Dẫn chứng) hoàn chỉnh, sẵn sàng làm bài kiểm tra cho hệ thống tìm kiếm thông tin. ```

In [ ]:
def build_claim_prompt(window: Dict[str, Any]) -> str:
    return f"""
Bạn là chuyên gia xây dựng benchmark truy xuất thông tin cho hệ thống RAG lịch sử Việt Nam.

Dữ liệu bên dưới là một đoạn trích từ sách {window["book_name"]}.
Nhiệm vụ của bạn là tạo các claim lịch sử để đánh giá semantic retrieval.

Yêu cầu:
- Chỉ sử dụng thông tin có trong đoạn trích.
- Không dùng kiến thức bên ngoài.
- Không copy nguyên văn toàn bộ câu dài từ đoạn trích.
- Claim phải ngắn, rõ, độc lập và có thể kiểm chứng.
- Claim phải có chủ ngữ rõ ràng.
- Claim nên liên quan đến nhân vật, sự kiện, thời gian, chức vụ, quan hệ hoặc kết quả lịch sử.
- Mỗi claim phải có source_excerpt.
- source_excerpt là đoạn ngắn nằm nguyên trong dữ liệu gốc, dùng làm căn cứ tạo claim.
- source_excerpt nên dài từ 20 đến 120 từ.
- source_excerpt không được tự viết lại.
- Tạo tối đa {MAX_CLAIMS_PER_WINDOW} claim.
- Không tạo claim nếu đoạn trích quá mơ hồ.
- Trả về JSON hợp lệ, không giải thích ngoài JSON.

Schema:
{{
  "claims": [
    {{
      "claim": "...",
      "source_excerpt": "...",
      "source_pages": [154],
      "difficulty": "easy|medium|hard",
      "explanation": "..."
    }}
  ]
}}

Dữ liệu:
{json.dumps(window, ensure_ascii=False, indent=2)}
""".strip()

In [ ]:
def extract_json(text: str) -> Dict[str, Any]:
    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise ValueError("Gemini response không có JSON hợp lệ.")
        return json.loads(match.group(0))


def call_gemini(client: genai.Client, prompt: str) -> Dict[str, Any]:
    max_retries = 6
    base_sleep = 10

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.2,
                    response_mime_type="application/json",
                ),
            )

            if not response.text:
                raise ValueError("Gemini trả về rỗng.")

            return extract_json(response.text)

        except Exception as e:
            error_text = str(e)

            is_retryable = (
                "503" in error_text
                or "UNAVAILABLE" in error_text
                or "429" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
                or "timeout" in error_text.lower()
            )

            if not is_retryable or attempt == max_retries - 1:
                raise e

            wait_time = base_sleep * (2 ** attempt)
            logging.warning(
                f"Gemini lỗi tạm thời. Retry {attempt + 1}/{max_retries} sau {wait_time}s"
            )
            time.sleep(wait_time)

    raise RuntimeError("Gemini failed after retries.")

In [ ]:
def build_claim_samples(
    gemini_output: Dict[str, Any],
    window: Dict[str, Any],
    start_index: int,
) -> List[Dict[str, Any]]:

    claims = gemini_output.get("claims", [])
    samples = []

    for obj in claims:
        claim = str(obj.get("claim", "")).strip()
        source_excerpt = str(obj.get("source_excerpt", "")).strip()
        difficulty = str(obj.get("difficulty", "medium")).strip().lower()
        explanation = str(obj.get("explanation", "")).strip()

        if not claim or not source_excerpt:
            continue

        if difficulty not in {"easy", "medium", "hard"}:
            difficulty = "medium"

        source_pages = obj.get("source_pages", window["pages"])

        if not isinstance(source_pages, list):
            source_pages = window["pages"]

        claim_id = f"claim_{start_index + len(samples):06d}"

        samples.append({
            "claim_id": claim_id,
            "claim": claim,
            "source_excerpt": source_excerpt,
            "source_pages": source_pages,
            "book_name": window["book_name"],
            "source_window_id": window["window_id"],
            "difficulty": difficulty,
            "explanation": explanation,
            "metadata": {
                "created_by": "gemini",
                "needs_human_review": True,
                "dataset_type": "multi_chunk_retrieval_benchmark"
            }
        })

    return samples

## sinh dataset

In [ ]:
api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError("Không lấy được GEMINI_API_KEY từ Colab Secrets.")

client = genai.Client(api_key=api_key)

claim_samples = []
failed_windows = []

for window in tqdm(windows, desc="Generating claims"):
    if len(claim_samples) >= TARGET_CLAIMS:
        break

    try:
        prompt = build_claim_prompt(window)
        output = call_gemini(client, prompt)

        samples = build_claim_samples(
            gemini_output=output,
            window=window,
            start_index=len(claim_samples) + 1,
        )

        claim_samples.extend(samples)

        # Lưu tạm sau mỗi window
        with Path("/content/temp_claims.jsonl").open("w", encoding="utf-8") as f:
            for row in claim_samples:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    except Exception as e:
        failed_windows.append({
            "window_id": window["window_id"],
            "pages": window["pages"],
            "error": str(e),
        })
        logging.warning(f"Lỗi ở {window['window_id']}: {e}")

    time.sleep(SLEEP_SECONDS)


with FAILED_JSON.open("w", encoding="utf-8") as f:
    json.dump(failed_windows, f, ensure_ascii=False, indent=2)

print("Generated claims:", len(claim_samples))
print("Failed windows:", len(failed_windows))

Generating claims:  97%|█████████▋| 75/77 [10:59<00:17,  8.79s/it]

Generated claims: 300
Failed windows: 0


## map

In [ ]:
# chec xem source_excerpt do gemini trả về và raw_text tìm được có cùng trang không
def page_overlap_score(source_pages: List[int], chunk_pages: List[int]) -> int:
    source_set = set(int(p) for p in source_pages if str(p).isdigit())
    chunk_set = set(int(p) for p in chunk_pages if str(p).isdigit())

    return len(source_set.intersection(chunk_set))


def exact_contains_match(
    source_excerpt: str,
    chunk_text: str,
) -> bool:
    source_clean = clean_for_match(source_excerpt)
    chunk_clean = clean_for_match(chunk_text)

    if len(source_clean) < MIN_EXACT_LEN:
        return False

    return source_clean in chunk_clean


def fuzzy_match_score(
    source_excerpt: str,
    chunk_text: str,
) -> float:
    source_clean = clean_for_match(source_excerpt)
    chunk_clean = clean_for_match(chunk_text)

    if not source_clean or not chunk_clean:
        return 0.0

    return fuzz.partial_ratio(source_clean, chunk_clean)


# gọi các hàm match để xem source_excerpt có trùng raw_text nào  trong các file chunking không
def find_matching_chunks(
    source_excerpt: str,
    source_pages: List[int],
    chunks: List[Dict[str, Any]],
    top_n: int = 3,
) -> List[Dict[str, Any]]:

    candidates = []
    tmp = 0;
    for chunk in chunks:
        chunk_text = chunk.get("raw_text", "")
        chunk_pages = chunk.get("pages", [])

        page_score = page_overlap_score(source_pages, chunk_pages)

        # Ưu tiên chunk cùng trang, nhưng không loại hoàn toàn chunk khác trang
        exact_match = exact_contains_match(source_excerpt, chunk_text)
        fuzzy_score = fuzzy_match_score(source_excerpt, chunk_text)

# hàm xem có trùng không (không cần trùng chính xác )
        final_score = fuzzy_score + (page_score * 10)
        tmp+=1


# trùng chính xác với nhau
        if exact_match:
            final_score += 30
# điểm số trùng lớn thì lấy
        if exact_match or final_score >= FUZZY_THRESHOLD:
            candidates.append({
                "chunk_id": chunk.get("chunk_id"),
                "pages": chunk_pages,
                "raw_text": chunk_text,
                "token_count": chunk.get("token_count", 0),
                "match_score": round(final_score, 4),
                "fuzzy_score": round(fuzzy_score, 4),
                "page_overlap": page_score,
                "exact_match": exact_match,
            })
        if tmp < 10:
            print(chunk["book_name"])
            print(chunk_pages)
            print(source_excerpt)
            print(chunk_text)
            print(f"final_score: {final_score} \n\n")

    candidates = sorted(
        candidates,
        key=lambda x: (
            x["exact_match"],
            x["page_overlap"],
            x["match_score"]
        ),
        reverse=True
    )

    return candidates[:top_n]

In [ ]:
def attach_qrels_to_claims(
    claim_samples: List[Dict[str, Any]],
    chunk_data: Dict[str, List[Dict[str, Any]]],
) -> List[Dict[str, Any]]:

    final_dataset = []

    for sample in tqdm(claim_samples, desc="Mapping source_excerpt to chunks"):
        qrels = {}

        for chunk_key, chunks in chunk_data.items():
            matches = find_matching_chunks(
                source_excerpt=sample["source_excerpt"],
                source_pages=sample["source_pages"],
                chunks=chunks,
                top_n=TOP_MATCHES_PER_CHUNK_SIZE,
            )

            qrels[chunk_key] = [
                {
                    "chunk_id": m["chunk_id"],
                    "pages": m["pages"],
                    "match_score": m["match_score"],
                    "fuzzy_score": m["fuzzy_score"],
                    "page_overlap": m["page_overlap"],
                    "exact_match": m["exact_match"],
                    "token_count": m["token_count"],
                    "text_preview": short_text(m["raw_text"], 350),
                }
                for m in matches
                if m.get("chunk_id")
            ]

        row = dict(sample)
        row["qrels"] = qrels

        row["qrels_status"] = {
            key: len(value)
            for key, value in qrels.items()
        }

        row["is_usable_for_all_chunk_sizes"] = all(
            len(qrels.get(key, [])) > 0
            for key in chunk_data.keys()
        )

        final_dataset.append(row)

    return final_dataset


final_dataset = attach_qrels_to_claims(claim_samples, chunk_data)

usable_all = [x for x in final_dataset if x["is_usable_for_all_chunk_sizes"]]

print("Total claims:", len(final_dataset))
print("Usable for all chunk sizes:", len(usable_all))
print("Example:")
print(json.dumps(final_dataset[0], ensure_ascii=False, indent=2)[:3000])

Mapping source_excerpt to chunks:   0%|          | 0/300 [00:00<?, ?it/s]

Đại Việt Sử Ký Toàn Thư
[154]
Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự... Cao Tông băng, bèn lên ngôi báu, ở ngôi 14 năm [1211-1224], truyền ngôi cho Chiêu Hoàng, sau bị Trần Thủ Độ giết, thọ 33 tuổi [1194-1226].
việc lớn của nước, quốc sử không thể không chép được. Có lẽ Lê Văn Hưu thấy đều gọi là Thọ lăng, cho là không đúng lễ nên bỏ đi, nhưng thế không phải là phép làm sử.

final_score: 50.0 


Đại Việt Sử Ký Toàn Thư
[154]
Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự... Cao Tông băng, bèn lên ngôi báu, ở ngôi 14 năm [1211-1224], truyền ngôi cho Chiêu Hoàng, sau bị Trần Thủ Độ giết, thọ 33 tuổi [1194-1226].
Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự. Lại đem thuyền rồng đi đón Trần thị. Anh Trần thị là Trần Tự Khánh cho rằng bấy giờ đương lúc loạn lạc, 

Mapping source_excerpt to chunks:   0%|          | 1/300 [00:03<16:59,  3.41s/it]

Đại Việt Sử Ký Toàn Thư
[154]
Vua mới lên ngôi, đem việc nước giao cho Thái uý Đàm Dĩ Mông. Dĩ Mông là người không có học thức, không có mưu thuật, lại nhu nhược không quyết đoán, chính sự ngày một đổ nát.
việc lớn của nước, quốc sử không thể không chép được. Có lẽ Lê Văn Hưu thấy đều gọi là Thọ lăng, cho là không đúng lễ nên bỏ đi, nhưng thế không phải là phép làm sử.

final_score: 53.51145038167938 


Đại Việt Sử Ký Toàn Thư
[154]
Vua mới lên ngôi, đem việc nước giao cho Thái uý Đàm Dĩ Mông. Dĩ Mông là người không có học thức, không có mưu thuật, lại nhu nhược không quyết đoán, chính sự ngày một đổ nát.
Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự. Lại đem thuyền rồng đi đón Trần thị. Anh Trần thị là Trần Tự Khánh cho rằng bấy giờ đương lúc loạn lạc, chưa đưa đi ngay được.
Sai sứ cáo phó với nhà Tống, nhà Tống sai người sang làm lễ tế điếu.

final_score: 55.79124579124579 


Đại Việt Sử Ký Toàn Thư
[154]
Vua 

Mapping source_excerpt to chunks:   1%|          | 2/300 [00:04<09:44,  1.96s/it]

...

Đại Việt Sử Ký Toàn Thư
[154]
Quý Ly sai hành khiển Hoàng Hối Khanh làm cát địa sứ. Hối Khanh đem các thôn như Cổ Lâu, gồm cả thảy 59 thôn trả cho nhà Minh. Quý Ly trách mắng, lăng nhục Hối Khanh vì trả lại đất nhiều quá.
việc lớn của nước, quốc sử không thể không chép được. Có lẽ Lê Văn Hưu thấy đều gọi là Thọ lăng, cho là không đúng lễ nên bỏ đi, nhưng thế không phải là phép làm sử.

final_score: 42.07119741100323 


Đại Việt Sử Ký Toàn Thư
[154]
Quý Ly sai hành khiển Hoàng Hối Khanh làm cát địa sứ. Hối Khanh đem các thôn như Cổ Lâu, gồm cả thảy 59 thôn trả cho nhà Minh. Quý Ly trách mắng, lăng nhục Hối Khanh vì trả lại đất nhiều quá.
Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự. Lại đem thuyền rồng đi đón Trần thị. Anh Trần thị là Trần Tự Khánh cho rằng bấy giờ đương lúc loạn lạc, chưa đưa đi ngay được.
Sai sứ cáo phó với nhà Tống, nhà Tống sai người sang làm lễ tế điếu.

final_score: 43.01075268817204 


Đạ

Mapping source_excerpt to chunks: 100%|█████████▉| 299/300 [09:48<00:02,  2.47s/it]

Đại Việt Sử Ký Toàn Thư
[154]
Hán Thương ra lệnh cho các quan viên không được đi hia, chỉ cho đi giày tơ gai sống. Lệ cũ đời trước: quan từ lục phẩm trở lên mới được đi hia.
việc lớn của nước, quốc sử không thể không chép được. Có lẽ Lê Văn Hưu thấy đều gọi là Thọ lăng, cho là không đúng lễ nên bỏ đi, nhưng thế không phải là phép làm sử.

final_score: 42.85714285714286 


Đại Việt Sử Ký Toàn Thư
[154]
Hán Thương ra lệnh cho các quan viên không được đi hia, chỉ cho đi giày tơ gai sống. Lệ cũ đời trước: quan từ lục phẩm trở lên mới được đi hia.
Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự. Lại đem thuyền rồng đi đón Trần thị. Anh Trần thị là Trần Tự Khánh cho rằng bấy giờ đương lúc loạn lạc, chưa đưa đi ngay được.
Sai sứ cáo phó với nhà Tống, nhà Tống sai người sang làm lễ tế điếu.

final_score: 47.773279352226716 


Đại Việt Sử Ký Toàn Thư
[154]
Hán Thương ra lệnh cho các quan viên không được đi hia, chỉ cho đi 

Mapping source_excerpt to chunks: 100%|██████████| 300/300 [09:49<00:00,  1.97s/it]

Total claims: 300
Usable for all chunk sizes: 297
Example:
{
  "claim_id": "claim_000001",
  "claim": "Lý Huệ Tông lên ngôi khi mới 16 tuổi và ở ngôi trong 14 năm.",
  "source_excerpt": "Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự... Cao Tông băng, bèn lên ngôi báu, ở ngôi 14 năm [1211-1224], truyền ngôi cho Chiêu Hoàng, sau bị Trần Thủ Độ giết, thọ 33 tuổi [1194-1226].",
  "source_pages": [
    154
  ],
  "book_name": "Đại Việt Sử Ký Toàn Thư",
  "source_window_id": "window_00001",
  "difficulty": "easy",
  "explanation": "Thông tin về tuổi lên ngôi và thời gian trị vì của vua Lý Huệ Tông được ghi chép trực tiếp trong đoạn trích.",
  "metadata": {
    "created_by": "gemini",
    "needs_human_review": true,
    "dataset_type": "multi_chunk_retrieval_benchmark"
  },
  "qrels": {
    "chunk_256": [
      {
        "chunk_id": "019e5654-af0f-703f-81e6-84b66b8ea660",
        "pages": [
          154
        ],
   

In [ ]:
def save_jsonl(rows: List[Dict[str, Any]], path: Path):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def save_json(rows: List[Dict[str, Any]], path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)


def flatten_for_csv(rows: List[Dict[str, Any]]) -> pd.DataFrame:
    flat_rows = []

    for row in rows:
        flat = {
            "claim_id": row["claim_id"],
            "claim": row["claim"],
            "source_excerpt": row["source_excerpt"],
            "source_pages": ",".join(map(str, row.get("source_pages", []))),
            "book_name": row["book_name"],
            "difficulty": row["difficulty"],
            "usable_all": row["is_usable_for_all_chunk_sizes"],
        }

        for key in ["chunk_256", "chunk_512", "chunk_1024"]:
            q = row["qrels"].get(key, [])
            flat[f"{key}_count"] = len(q)
            flat[f"{key}_top_chunk_id"] = q[0]["chunk_id"] if q else ""
            flat[f"{key}_top_score"] = q[0]["match_score"] if q else ""
            flat[f"{key}_exact_match"] = q[0]["exact_match"] if q else ""

        flat_rows.append(flat)

    return pd.DataFrame(flat_rows)


save_jsonl(final_dataset, OUTPUT_JSONL)
save_json(final_dataset, OUTPUT_JSON)

df = flatten_for_csv(final_dataset)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(OUTPUT_JSONL)
print(OUTPUT_JSON)
print(OUTPUT_CSV)

df.head()

Saved:
/content/dvsk_embedding_benchmark_qrels.jsonl
/content/dvsk_embedding_benchmark_qrels.json
/content/dvsk_embedding_benchmark_qrels.csv


,claim_id,claim,source_excerpt,source_pages,book_name,difficulty,usable_all,chunk_256_count,chunk_256_top_chunk_id,chunk_256_top_score,chunk_256_exact_match,chunk_512_count,chunk_512_top_chunk_id,chunk_512_top_score,chunk_512_exact_match,chunk_1024_count,chunk_1024_top_chunk_id,chunk_1024_top_score,chunk_1024_exact_match
0,claim_000001,Lý Huệ Tông lên ngôi khi mới 16 tuổi và ở ngôi...,Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấ...,154,Đại Việt Sử Ký Toàn Thư,easy,True,2,019e5654-af0f-703f-81e6-84b66b8ea660,83.2794,False,1,019e5677-399e-7e71-b07f-dbce7e195260,83.2794,False,1,019e5742-d5f1-77a4-be9b-af7fe96d1fcd,83.2794,False
1,claim_000002,Thái uý Đàm Dĩ Mông bị đánh giá là người thiếu...,"Vua mới lên ngôi, đem việc nước giao cho Thái ...",154,Đại Việt Sử Ký Toàn Thư,medium,True,1,019e5654-af27-77d3-bdbd-bd8dca5c217c,140.0,True,1,019e5677-39b5-7c9e-98a6-60e40145e716,140.0,True,1,019e5742-d797-7b17-8bd3-d453fa6754c7,140.0000,True
2,claim_000003,"Lý Huệ Tông từng mắc bệnh tâm thần, tự xưng là...","Mùa xuân, tháng 3, vua dần dần phát điên, có k...",155,Đại Việt Sử Ký Toàn Thư,medium,True,1,019e5654-af67-726a-9b8b-b67bdab20534,140.0,True,1,019e5677-39ef-71bd-b844-7b6bc7c45313,140.0,True,1,019e5742-d7ff-7f68-b7f1-5f08edd5a248,140.0000,True
3,claim_000004,Trần Tự Khánh được phong làm Thái uý phụ chính...,"Mùa đông, tháng 12, sách phong [Thuận Trinh] p...",155,Đại Việt Sử Ký Toàn Thư,hard,True,1,019e5654-af57-783b-bf58-276e60e0861c,140.0,True,1,019e5677-39e7-7a4c-a3d9-0aa5826dae69,140.0,True,1,019e5742-d7e1-7897-8606-6a6753225f14,140.0000,True
4,claim_000005,Vua Huệ Tông đã chia đất nước thành 24 lộ vào ...,"Nhâm Ngọ, [Kiến Gia] năm thứ 12 [1222], (Tống ...",156,Đại Việt Sử Ký Toàn Thư,easy,True,1,019e5654-af90-74df-ae40-0687e081efb3,140.0,True,1,019e5677-3a17-7981-b327-847c2551882f,140.0,True,1,019e5742-d820-7de6-9b32-023a21d6ae5c,140.0000,True


In [ ]:
USABLE_JSONL = Path("/content/dvsk_embedding_benchmark_qrels_usable_all.jsonl")
USABLE_JSON = Path("/content/dvsk_embedding_benchmark_qrels_usable_all.json")
USABLE_CSV = Path("/content/dvsk_embedding_benchmark_qrels_usable_all.csv")

save_jsonl(usable_all, USABLE_JSONL)
save_json(usable_all, USABLE_JSON)

usable_df = flatten_for_csv(usable_all)
usable_df.to_csv(USABLE_CSV, index=False, encoding="utf-8-sig")

print("Usable all:", len(usable_all))
print(USABLE_JSONL)
print(USABLE_JSON)
print(USABLE_CSV)

files.download(str(OUTPUT_JSONL))
files.download(str(OUTPUT_JSON))
files.download(str(OUTPUT_CSV))

files.download(str(USABLE_JSONL))
files.download(str(USABLE_JSON))
files.download(str(USABLE_CSV))

files.download(str(FAILED_JSON))

Usable all: 297
/content/dvsk_embedding_benchmark_qrels_usable_all.jsonl
/content/dvsk_embedding_benchmark_qrels_usable_all.json
/content/dvsk_embedding_benchmark_qrels_usable_all.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 ``` {
  "claim_id": "claim_000001",
  "claim": "Huệ Tông là con trưởng của Cao Tông.",
  "source_excerpt": "Huệ Tông Hoàng Đế tên huý là Sảm, con trưởng của Cao Tông...",
  "source_pages": [154],
  "book_name": "Đại Việt Sử Ký Toàn Thư",
  "difficulty": "easy",
  "qrels": {
    "chunk_256": [
      {
        "chunk_id": "019e...",
        "pages": [154],
        "match_score": 100.0,
        "exact_match": true
      }
    ],
    "chunk_512": [
      {
        "chunk_id": "019f...",
        "pages": [154],
        "match_score": 98.2,
        "exact_match": true
      }
    ],
    "chunk_1024": [
      {
        "chunk_id": "020a...",
        "pages": [154, 155],
        "match_score": 96.7,
        "exact_match": true
      }
    ]
  }
} ```

In [ ]:
# !pip install -q qdrant-client sentence-transformers pandas tqdm numpy underthesea

# đánh giá

In [ ]:
import os
import re
import json
import math
import gc
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Any, Dict, List

from tqdm import tqdm
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

from underthesea import word_tokenize
from google.colab import files


DATASET_PATH = Path("/content/dvsk_embedding_benchmark_qrels_usable_all.jsonl")

SUMMARY_CSV = Path("/content/embedding_eval_summary_multi_qrels.csv")
DETAIL_CSV = Path("/content/embedding_eval_details_multi_qrels.csv")


QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

if not QDRANT_URL or not QDRANT_API_KEY:
    raise RuntimeError("Thiếu QDRANT_URL hoặc QDRANT_API_KEY trong biến môi trường Colab.")

qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

In [ ]:
MODEL_CONFIGS = [
    {
        "name": "bkai_256",
        "hf_model": "bkai-foundation-models/vietnamese-bi-encoder",
        "collection": "history_bkai_chunk_256",
        "chunk_key": "chunk_256",
        "needs_word_segmentation": True,
    },
    {
        "name": "dangvantuan_256",
        "hf_model": "dangvantuan/vietnamese-embedding",
        "collection": "history_dangvantuan_chunk_256",
        "chunk_key": "chunk_256",
        "needs_word_segmentation": True,
    },
    {
        "name": "halong_256",
        "hf_model": "hiieu/halong_embedding",
        "collection": "history_halong_chunk_256",
        "chunk_key": "chunk_256",
    },
    {
        "name": "bge_m3_256",
        "hf_model": "BAAI/bge-m3",
        "collection": "history_bge_m3_chunk_256",
        "chunk_key": "chunk_256",
    },
    {
        "name": "aiteamvn_256",
        "hf_model": "AITeamVN/Vietnamese_Embedding",
        "collection": "history_aiteamvn_chunk_256",
        "chunk_key": "chunk_256",
    },
    {
        "name": "halong_512",
        "hf_model": "hiieu/halong_embedding",
        "collection": "history_halong_chunk_512",
        "chunk_key": "chunk_512",
    },
    {
        "name": "bge_m3_512",
        "hf_model": "BAAI/bge-m3",
        "collection": "history_bge_m3_chunk_512",
        "chunk_key": "chunk_512",
    },
    {
        "name": "aiteamvn_512",
        "hf_model": "AITeamVN/Vietnamese_Embedding",
        "collection": "history_aiteamvn_chunk_512",
        "chunk_key": "chunk_512",
    },

    {
        "name": "bge_m3_1024",
        "hf_model": "BAAI/bge-m3",
        "collection": "history_bge_m3_chunk_1024",
        "chunk_key": "chunk_1024",
    },
    {
        "name": "aiteamvn_1024",
        "hf_model": "AITeamVN/Vietnamese_Embedding",
        "collection": "history_aiteamvn_chunk_1024",
        "chunk_key": "chunk_1024",
    },
]

In [ ]:
def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return rows


dataset = load_jsonl(DATASET_PATH)

print("Total samples:", len(dataset))
print(json.dumps(dataset[0], ensure_ascii=False, indent=2)[:2500])

Total samples: 297
{
  "claim_id": "claim_000001",
  "claim": "Lý Huệ Tông lên ngôi khi mới 16 tuổi và ở ngôi trong 14 năm.",
  "source_excerpt": "Hoàng thái tử Sảm lên ngôi ở trước linh cữu bấy giờ mới 16 tuổi. Tôn mẹ là Đàm thị là Hoàng thái hậu, cùng nghe chính sự... Cao Tông băng, bèn lên ngôi báu, ở ngôi 14 năm [1211-1224], truyền ngôi cho Chiêu Hoàng, sau bị Trần Thủ Độ giết, thọ 33 tuổi [1194-1226].",
  "source_pages": [
    154
  ],
  "book_name": "Đại Việt Sử Ký Toàn Thư",
  "source_window_id": "window_00001",
  "difficulty": "easy",
  "explanation": "Thông tin về tuổi lên ngôi và thời gian trị vì của vua Lý Huệ Tông được ghi chép trực tiếp trong đoạn trích.",
  "metadata": {
    "created_by": "gemini",
    "needs_human_review": true,
    "dataset_type": "multi_chunk_retrieval_benchmark"
  },
  "qrels": {
    "chunk_256": [
      {
        "chunk_id": "019e5654-af0f-703f-81e6-84b66b8ea660",
        "pages": [
          154
        ],
        "match_score": 83.2794,
        "fu

In [ ]:
def get_positive_chunk_ids(sample: Dict[str, Any], chunk_key: str) -> List[str]:
    qrels = sample.get("qrels", {}).get(chunk_key, [])

    ids = []

    for item in qrels:
        cid = item.get("chunk_id")
        if cid:
            ids.append(str(cid))

    return list(dict.fromkeys(ids))


for key in ["chunk_256", "chunk_512", "chunk_1024"]:
    counts = [len(get_positive_chunk_ids(row, key)) for row in dataset]
    print(key)
    print("samples with qrels:", sum(c > 0 for c in counts))
    print("avg positive chunks:", sum(counts) / len(counts))

chunk_256
samples with qrels: 297
avg positive chunks: 1.063973063973064
chunk_512
samples with qrels: 297
avg positive chunks: 1.0606060606060606
chunk_1024
samples with qrels: 297
avg positive chunks: 1.0505050505050506


In [ ]:
def recall_at_k(retrieved_ids: List[str], positive_ids: List[str], k: int) -> float:
    top_k = retrieved_ids[:k]
    return 1.0 if any(pid in top_k for pid in positive_ids) else 0.0


def reciprocal_rank_at_k(retrieved_ids: List[str], positive_ids: List[str], k: int) -> float:
    for idx, rid in enumerate(retrieved_ids[:k], start=1):
        if rid in positive_ids:
            return 1.0 / idx
    return 0.0


def dcg_at_k(retrieved_ids: List[str], positive_ids: List[str], k: int) -> float:
    dcg = 0.0

    for idx, rid in enumerate(retrieved_ids[:k], start=1):
        rel = 1.0 if rid in positive_ids else 0.0
        if rel > 0:
            dcg += rel / math.log2(idx + 1)

    return dcg


def ndcg_at_k(retrieved_ids: List[str], positive_ids: List[str], k: int) -> float:
    dcg = dcg_at_k(retrieved_ids, positive_ids, k)

    ideal_hits = min(len(positive_ids), k)

    if ideal_hits == 0:
        return 0.0

    idcg = sum(
        1.0 / math.log2(i + 1)
        for i in range(1, ideal_hits + 1)
    )

    return dcg / idcg if idcg > 0 else 0.0

In [ ]:
def normalize_query(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def preprocess_query(text: str, config: Dict[str, Any]) -> str:
    text = normalize_query(text)

    if config.get("needs_word_segmentation", False):
        return word_tokenize(text, format="text")

    return text

In [ ]:
def search_qdrant(
    client: QdrantClient,
    collection_name: str,
    query_vector: List[float],
    limit: int = 10,
) -> List[Dict[str, Any]]:

    result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=limit,
        with_payload=True,
        with_vectors=False,
    )

    rows = []

    for point in result.points:
        payload = point.payload or {}

        chunk_id = (
            payload.get("chunk_id")
            or payload.get("id")
            or str(point.id)
        )

        rows.append({
            "chunk_id": str(chunk_id),
            "score": float(point.score),
            "pages": payload.get("pages", []),
            "text_preview": str(payload.get("raw_text", ""))[:300],
        })

    return rows

In [ ]:
def encode_queries(
    model: SentenceTransformer,
    queries: List[str],
    batch_size: int = 16,
) -> np.ndarray:

    embeddings = model.encode(
        queries,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    return embeddings

### đánh giá model

In [ ]:
def evaluate_one_model(
    config: Dict[str, Any],
    dataset: List[Dict[str, Any]],
    top_k: int = 10,
    batch_size: int = 16,
) -> Dict[str, Any]:

    model_name = config["name"]
    hf_model = config["hf_model"]
    collection = config["collection"]
    chunk_key = config["chunk_key"]
    trust_remote_code = bool(config.get("trust_remote_code", False))

    print("\n" + "=" * 80)
    print("Evaluating:", model_name)
    print("HF model:", hf_model)
    print("Collection:", collection)
    print("Qrels:", chunk_key)

    eval_rows = []

    usable_dataset = [
        row for row in dataset
        if len(get_positive_chunk_ids(row, chunk_key)) > 0
    ]

    print("Usable samples:", len(usable_dataset))

    model = SentenceTransformer(
        hf_model,
        trust_remote_code=trust_remote_code,
    )

    queries = [
        preprocess_query(row["claim"], config)
        for row in usable_dataset
    ]

    query_vectors = encode_queries(
        model=model,
        queries=queries,
        batch_size=batch_size,
    )

    for row, vector in tqdm(
        list(zip(usable_dataset, query_vectors)),
        desc=f"Searching {model_name}"
    ):
        positive_ids = get_positive_chunk_ids(row, chunk_key)

        try:
            search_results = search_qdrant(
                client=qdrant,
                collection_name=collection,
                query_vector=vector.tolist(),
                limit=top_k,
            )

            retrieved_ids = [x["chunk_id"] for x in search_results]

            eval_rows.append({
                "model_name": model_name,
                "collection": collection,
                "chunk_key": chunk_key,
                "claim_id": row["claim_id"],
                "claim": row["claim"],
                "difficulty": row.get("difficulty", "medium"),
                "positive_chunk_ids": positive_ids,
                "retrieved_ids": retrieved_ids,
                "top_scores": [x["score"] for x in search_results],
                "recall@1": recall_at_k(retrieved_ids, positive_ids, 1),
                "recall@3": recall_at_k(retrieved_ids, positive_ids, 3),
                "recall@5": recall_at_k(retrieved_ids, positive_ids, 5),
                "recall@10": recall_at_k(retrieved_ids, positive_ids, 10),
                "mrr@10": reciprocal_rank_at_k(retrieved_ids, positive_ids, 10),
                "ndcg@10": ndcg_at_k(retrieved_ids, positive_ids, 10),
                "error": "",
            })

        except Exception as e:
            eval_rows.append({
                "model_name": model_name,
                "collection": collection,
                "chunk_key": chunk_key,
                "claim_id": row["claim_id"],
                "claim": row["claim"],
                "difficulty": row.get("difficulty", "medium"),
                "positive_chunk_ids": positive_ids,
                "retrieved_ids": [],
                "top_scores": [],
                "recall@1": 0.0,
                "recall@3": 0.0,
                "recall@5": 0.0,
                "recall@10": 0.0,
                "mrr@10": 0.0,
                "ndcg@10": 0.0,
                "error": str(e),
            })

    details_df = pd.DataFrame(eval_rows)

    summary = {
        "model_name": model_name,
        "collection": collection,
        "chunk_key": chunk_key,
        "num_queries": len(details_df),
        "recall@1": details_df["recall@1"].mean(),
        "recall@3": details_df["recall@3"].mean(),
        "recall@5": details_df["recall@5"].mean(),
        "recall@10": details_df["recall@10"].mean(),
        "mrr@10": details_df["mrr@10"].mean(),
        "ndcg@10": details_df["ndcg@10"].mean(),
        "error_count": int((details_df["error"] != "").sum()),
    }

    del model
    gc.collect()

    return {
        "summary": summary,
        "details": details_df,
    }

### main

In [ ]:
all_summaries = []
all_details = []

for config in MODEL_CONFIGS:
    result = evaluate_one_model(
        config=config,
        dataset=dataset,
        top_k=10,
        batch_size=16,
    )

    all_summaries.append(result["summary"])
    all_details.append(result["details"])

    summary_df = pd.DataFrame(all_summaries)
    details_df = pd.concat(all_details, ignore_index=True)

    summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")
    details_df.to_csv(DETAIL_CSV, index=False, encoding="utf-8-sig")

    display(summary_df.sort_values(
        by=["recall@5", "mrr@10", "ndcg@10"],
        ascending=False
    ))


Evaluating: bkai_256
HF model: bkai-foundation-models/vietnamese-bi-encoder
Collection: history_bkai_chunk_256
Qrels: chunk_256
Usable samples: 297


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Searching bkai_256: 100%|██████████| 297/297 [00:34<00:00,  8.58it/s]


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
0,bkai_256,history_bkai_chunk_256,chunk_256,297,0.255892,0.478114,0.535354,0.636364,0.381665,0.440378,0


...


Evaluating: aiteamvn_1024
HF model: AITeamVN/Vietnamese_Embedding
Collection: history_aiteamvn_chunk_1024
Qrels: chunk_1024
Usable samples: 297


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Searching aiteamvn_1024: 100%|██████████| 297/297 [00:34<00:00,  8.61it/s]


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
3,bge_m3_256,history_bge_m3_chunk_256,chunk_256,297,0.356902,0.643098,0.764310,0.851852,0.522682,0.596870,0
4,aiteamvn_256,history_aiteamvn_chunk_256,chunk_256,297,0.367003,0.649832,0.757576,0.868687,0.529843,0.603282,0
6,bge_m3_512,history_bge_m3_chunk_512,chunk_512,297,0.363636,0.636364,0.744108,0.814815,0.516842,0.585759,0
7,aiteamvn_512,history_aiteamvn_chunk_512,chunk_512,297,0.363636,0.619529,0.740741,0.831650,0.516183,0.588990,0
8,bge_m3_1024,history_bge_m3_chunk_1024,chunk_1024,297,0.346801,0.602694,0.693603,0.791246,0.497970,0.565317,0
5,halong_512,history_halong_chunk_512,chunk_512,297,0.319865,0.538721,0.656566,0.740741,0.447997,0.515263,0
9,aiteamvn_1024,history_aiteamvn_chunk_1024,chunk_1024,297,0.336700,0.569024,0.653199,0.764310,0.473830,0.542257,0
2,halong_256,history_halong_chunk_256,chunk_256,297,0.323232,0.552189,0.636364,0.764310,0.462133,0.529423,0
0,bkai_256,history_bkai_chunk_256,chunk_256,297,0.255892,0.478114,0.535354,0.636364,0.381665,0.440378,0
1,dangvantuan_256,history_dangvantuan_chunk_256,chunk_256,297,0.245791,0.434343,0.508418,0.609428,0.355821,0.411251,0


In [ ]:
summary_df = pd.DataFrame(all_summaries)

summary_df = summary_df.sort_values(
    by=["chunk_key", "recall@5", "mrr@10", "ndcg@10"],
    ascending=[True, False, False, False]
)

summary_df.to_csv(SUMMARY_CSV, index=False, encoding="utf-8-sig")

summary_df

,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
8,bge_m3_1024,history_bge_m3_chunk_1024,chunk_1024,297,0.346801,0.602694,0.693603,0.791246,0.497970,0.565317,0
9,aiteamvn_1024,history_aiteamvn_chunk_1024,chunk_1024,297,0.336700,0.569024,0.653199,0.764310,0.473830,0.542257,0
3,bge_m3_256,history_bge_m3_chunk_256,chunk_256,297,0.356902,0.643098,0.764310,0.851852,0.522682,0.596870,0
4,aiteamvn_256,history_aiteamvn_chunk_256,chunk_256,297,0.367003,0.649832,0.757576,0.868687,0.529843,0.603282,0
2,halong_256,history_halong_chunk_256,chunk_256,297,0.323232,0.552189,0.636364,0.764310,0.462133,0.529423,0
0,bkai_256,history_bkai_chunk_256,chunk_256,297,0.255892,0.478114,0.535354,0.636364,0.381665,0.440378,0
1,dangvantuan_256,history_dangvantuan_chunk_256,chunk_256,297,0.245791,0.434343,0.508418,0.609428,0.355821,0.411251,0
6,bge_m3_512,history_bge_m3_chunk_512,chunk_512,297,0.363636,0.636364,0.744108,0.814815,0.516842,0.585759,0
7,aiteamvn_512,history_aiteamvn_chunk_512,chunk_512,297,0.363636,0.619529,0.740741,0.831650,0.516183,0.588990,0
5,halong_512,history_halong_chunk_512,chunk_512,297,0.319865,0.538721,0.656566,0.740741,0.447997,0.515263,0


# idk

In [ ]:
# @title
for chunk_key in ["chunk_256", "chunk_512", "chunk_1024"]:
    print("\n" + "=" * 80)
    print(chunk_key)

    display(
        summary_df[summary_df["chunk_key"] == chunk_key]
        .sort_values(
            by=["recall@5", "mrr@10", "ndcg@10"],
            ascending=False
        )
    )


chunk_256


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
3,bge_m3_256,history_bge_m3_chunk_256,chunk_256,297,0.356902,0.643098,0.764310,0.851852,0.522682,0.596870,0
4,aiteamvn_256,history_aiteamvn_chunk_256,chunk_256,297,0.367003,0.649832,0.757576,0.868687,0.529843,0.603282,0
2,halong_256,history_halong_chunk_256,chunk_256,297,0.323232,0.552189,0.636364,0.764310,0.462133,0.529423,0
0,bkai_256,history_bkai_chunk_256,chunk_256,297,0.255892,0.478114,0.535354,0.636364,0.381665,0.440378,0
1,dangvantuan_256,history_dangvantuan_chunk_256,chunk_256,297,0.245791,0.434343,0.508418,0.609428,0.355821,0.411251,0



chunk_512


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
6,bge_m3_512,history_bge_m3_chunk_512,chunk_512,297,0.363636,0.636364,0.744108,0.814815,0.516842,0.585759,0
7,aiteamvn_512,history_aiteamvn_chunk_512,chunk_512,297,0.363636,0.619529,0.740741,0.831650,0.516183,0.588990,0
5,halong_512,history_halong_chunk_512,chunk_512,297,0.319865,0.538721,0.656566,0.740741,0.447997,0.515263,0



chunk_1024


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
8,bge_m3_1024,history_bge_m3_chunk_1024,chunk_1024,297,0.346801,0.602694,0.693603,0.791246,0.49797,0.565317,0
9,aiteamvn_1024,history_aiteamvn_chunk_1024,chunk_1024,297,0.336700,0.569024,0.653199,0.764310,0.47383,0.542257,0


In [ ]:
# @title
for chunk_key in ["chunk_256", "chunk_512", "chunk_1024"]:
    print("\n" + "=" * 80)
    print(chunk_key)

    display(
        summary_df[summary_df["chunk_key"] == chunk_key]
        .sort_values(
            by=["recall@5", "mrr@10", "ndcg@10"],
            ascending=False
        )
    )


chunk_256


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
4,aiteamvn_256,history_aiteamvn_chunk_256,chunk_256,198,0.313131,0.676768,0.777778,0.888889,0.508882,0.591908,0
3,bge_m3_256,history_bge_m3_chunk_256,chunk_256,198,0.313131,0.651515,0.772727,0.868687,0.503245,0.585383,0
2,halong_256,history_halong_chunk_256,chunk_256,198,0.313131,0.590909,0.661616,0.762626,0.462039,0.528407,0
0,bkai_256,history_bkai_chunk_256,chunk_256,198,0.247475,0.525253,0.575758,0.661616,0.393879,0.458133,0
1,dangvantuan_256,history_dangvantuan_chunk_256,chunk_256,198,0.186869,0.308081,0.388889,0.494949,0.271178,0.319218,0



chunk_512


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
6,bge_m3_512,history_bge_m3_chunk_512,chunk_512,198,0.343434,0.601010,0.747475,0.853535,0.506035,0.584962,0
7,aiteamvn_512,history_aiteamvn_chunk_512,chunk_512,198,0.373737,0.616162,0.722222,0.838384,0.520717,0.592898,0
5,halong_512,history_halong_chunk_512,chunk_512,198,0.328283,0.540404,0.676768,0.757576,0.461253,0.527468,0



chunk_1024


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
8,bge_m3_1024,history_bge_m3_chunk_1024,chunk_1024,198,0.343434,0.616162,0.717172,0.80303,0.503547,0.571963,0
9,aiteamvn_1024,history_aiteamvn_chunk_1024,chunk_1024,198,0.383838,0.606061,0.691919,0.80303,0.517126,0.581624,0


In [ ]:
# @title
for chunk_key in ["chunk_256", "chunk_512", "chunk_1024"]:
    print("\n" + "=" * 80)
    print(chunk_key)

    display(
        summary_df[summary_df["chunk_key"] == chunk_key]
        .sort_values(
            by=["recall@5", "mrr@10", "ndcg@10"],
            ascending=False
        )
    )


chunk_256


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
3,bge_m3_256,history_bge_m3_chunk_256,chunk_256,100,0.26,0.59,0.70,0.82,0.445762,0.520886,0
4,aiteamvn_256,history_aiteamvn_chunk_256,chunk_256,100,0.22,0.56,0.68,0.80,0.410996,0.486695,0
2,halong_256,history_halong_chunk_256,chunk_256,100,0.23,0.47,0.56,0.66,0.367381,0.423954,0
0,bkai_256,history_bkai_chunk_256,chunk_256,100,0.18,0.41,0.47,0.55,0.304135,0.357596,0
1,dangvantuan_256,history_dangvantuan_chunk_256,chunk_256,100,0.16,0.40,0.45,0.55,0.281163,0.334261,0



chunk_512


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
6,bge_m3_512,history_bge_m3_chunk_512,chunk_512,100,0.25,0.55,0.68,0.79,0.425992,0.504167,0
7,aiteamvn_512,history_aiteamvn_chunk_512,chunk_512,100,0.27,0.53,0.68,0.72,0.418111,0.479437,0
5,halong_512,history_halong_chunk_512,chunk_512,100,0.25,0.45,0.57,0.66,0.372440,0.431719,0



chunk_1024


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
8,bge_m3_1024,history_bge_m3_chunk_1024,chunk_1024,100,0.21,0.47,0.61,0.72,0.374591,0.448236,0
9,aiteamvn_1024,history_aiteamvn_chunk_1024,chunk_1024,100,0.27,0.50,0.60,0.71,0.413020,0.477859,0


In [ ]:
# @title
for chunk_key in ["chunk_256", "chunk_512", "chunk_1024"]:
    print("\n" + "=" * 80)
    print(chunk_key)

    display(
        summary_df[summary_df["chunk_key"] == chunk_key]
        .sort_values(
            by=["recall@5", "mrr@10", "ndcg@10"],
            ascending=False
        )
    )


chunk_256


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
3,bge_m3_256,history_bge_m3_chunk_256,chunk_256,299,0.331104,0.662207,0.769231,0.859532,0.511941,0.590574,0
4,aiteamvn_256,history_aiteamvn_chunk_256,chunk_256,299,0.384615,0.665552,0.755853,0.856187,0.541545,0.609794,0
2,halong_256,history_halong_chunk_256,chunk_256,299,0.344482,0.581940,0.658863,0.765886,0.476411,0.541238,0
0,bkai_256,history_bkai_chunk_256,chunk_256,299,0.260870,0.501672,0.561873,0.652174,0.391744,0.452157,0
1,dangvantuan_256,history_dangvantuan_chunk_256,chunk_256,299,0.230769,0.418060,0.498328,0.588629,0.342147,0.397334,0



chunk_512


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
6,bge_m3_512,history_bge_m3_chunk_512,chunk_512,299,0.394649,0.642140,0.765886,0.849498,0.544377,0.614115,0
7,aiteamvn_512,history_aiteamvn_chunk_512,chunk_512,299,0.377926,0.635452,0.752508,0.856187,0.530522,0.605423,0
5,halong_512,history_halong_chunk_512,chunk_512,299,0.331104,0.561873,0.658863,0.745819,0.463081,0.528853,0



chunk_1024


,model_name,collection,chunk_key,num_queries,recall@1,recall@3,recall@5,recall@10,mrr@10,ndcg@10,error_count
8,bge_m3_1024,history_bge_m3_chunk_1024,chunk_1024,299,0.351171,0.615385,0.698997,0.812709,0.505414,0.577041,0
9,aiteamvn_1024,history_aiteamvn_chunk_1024,chunk_1024,299,0.361204,0.581940,0.655518,0.782609,0.490858,0.558963,0


In [ ]:
files.download(str(SUMMARY_CSV))
files.download(str(DETAIL_CSV))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>